In [2]:
from functools import partial
import jax
import jax.numpy as jnp
from typing import Optional
from flax import struct


@jax.jit
def sample_splitting_plane(rng: jax.Array, data: jax.Array):
    n_samples, n_features = data.shape
    normal = jax.random.normal(rng, shape=(n_features,))
    normal = normal / jnp.linalg.norm(normal)
    distances = jax.vmap(jnp.dot, in_axes=(0, None))(data, normal)
    intercept = jax.random.uniform(rng, minval=distances.min(), maxval=distances.max())
    return normal, intercept, distances - intercept


class Node(struct.PyTreeNode):
    data: jax.Array
    depth: int = 0
    normal: Optional[jax.Array] = None
    intercept: Optional[jax.Array] = None
    left_child: Optional["Node"] = None
    right_child: Optional["Node"] = None

    def split(self, rng: jax.Array):
        normal, intercept, distances = sample_splitting_plane(rng, self.data)
        left_data = self.data[distances < 0]
        right_data = self.data[distances >= 0]

        return self.replace(
            normal=normal,
            intercept=intercept,
            left_child=Node(left_data, self.depth + 1),
            right_child=Node(right_data, self.depth + 1),
        )

In [ ]:
rng = jax.random.PRNGKey(42)
data = jax.random.normal(rng, (32, 2))
tree = IsolationTree.fit(rng, data, n_components)

X, Y = jnp.meshgrid(jnp.linspace(-5, 5, 1000), jnp.linspace(-5, 5, 1000))
coord = jnp.stack([X.flatten(), Y.flatten()]).T
data_paths = jax.vmap(tree.path)(data)
region_paths = jax.vmap(tree.path)(coord)

for depth, (data_path_ids, region_paths_ids) in enumerate(
    zip(data_paths[:, 1:].T, region_paths[:, 1:].T)
):
    print(f"data reached node {data_path_ids}")
    plt.figure(figsize=(6, 6))
    # scatter data color coded by region
    plt.scatter(
        data[:, 0], data[:, 1],s=200, c=data_path_ids,cmap="tab20", edgecolors="k",
        vmin=region_paths_ids.min(), vmax=region_paths_ids.min() + 19,
    )
    plt.scatter(data[:, 0], data[:, 1], marker="x", c="k", s=10)
    # imshow node regions
    plt.imshow(
        region_paths_ids.reshape(1000, 1000), cmap="tab20", vmin=region_paths_ids.min(),
        vmax=region_paths_ids.min() + 19, extent=(-5, 5, 5, -5),
    )
    plt.colorbar()
    plt.show()